ЧТобы посчитать TF-IDF метрику, нам нужно вытащить из датасета посты. Также, нужно очистить данные от выбросов, поэтому из исходного data.json сделаем две таблицы: users_df -- очищенная таблица с пользователями, и posts_df -- посты этих пользователей.

In [13]:
import json
import pandas as pd

with open('data.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

users_list =[]
for u in raw_data:
    # фильтруем пустые страницы
    if u['friends_num'] > 0 and u['self_posts_num'] > 0:
        users_list.append({
            'user_id': u['id'],
            'city': u['city'],
            'age': u['age'],
            'gender': u['gender'],
            'friends_num': u['friends_num'],
            'groups_num': u['groups_num'],
            'self_posts_num': u['self_posts_num']
        })

users_df = pd.DataFrame(users_list)

#убираем выбросы(блогеры с болоьшим кол-вом друзей и т.п.) и берем 95-й перцентиль
q95 = users_df['friends_num'].quantile(0.95)
users_df = users_df[users_df['friends_num'] <= q95]

#все посты в отдельную таблицу
posts_list =[]
for u in raw_data:
    if u['id'] in users_df['user_id'].values:
        for post_id, post_data in u['posts'].items():
            text = post_data['text'].strip()
            if len(text) > 10:
                posts_list.append({
                    'user_id': u['id'],
                    'post_id': post_id,
                    'text': text
                })

posts_df = pd.DataFrame(posts_list)

print(f"Осталось пользователей: {len(users_df)}")
print(f"Собрано постов для ML: {len(posts_df)}")

Осталось пользователей: 129
Собрано постов для ML: 5555


In [14]:
pip install datasets scikit-learn

Для обучения ML модели возьмем датасет k1tub/sentiment_dataset, представляющий собой тексты из разных источников(новости, отзывы, посты в соц.сетях), размеченных по тональности. В датасете метка 0 означает нейтрально, 1 -- это позитив и 2 -- это негатив.

In [15]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

dataset = load_dataset("k1tub/sentiment_dataset")

df = dataset['train'].to_pandas()

#оставим только позитивные и негативные, без нейтральных
df = df[df['label'].isin([1, 2])]

#тепрь 1 - хорошо, 0 - плохо
df['label'] = df['label'].replace(2, 0)


df_train = df.sample(100000, random_state=42)

train_texts = df_train['text'].tolist()
train_labels = df_train['label'].tolist()

print("Векторизация (то есть TF-IDF)")
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_texts)

print("Обучаем логистическую регрессию")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_labels)



Векторизация (то есть TF-IDF)
Обучаем логистическую регрессию


LogisticRegression(max_iter=1000)

Теперь у нас есть обученная model и vectorizer. Берем посыт (posts_df), переводим их в векторы и просим модель угадать, позитив это или негатив.

In [16]:
X_vk = vectorizer.transform(posts_df['text']) #посты из вк в векторы

posts_df['sentiment'] = model.predict(X_vk)

#группируем по user_id и считаем среднее (это доля позитивных постов)
sentiment_scores = posts_df.groupby('user_id')['sentiment'].mean().reset_index()
sentiment_scores.rename(columns={'sentiment': 'positive_posts_percent'}, inplace=True)

users_df = users_df.merge(sentiment_scores, on='user_id', how='left')

#если у кого-то все посты отфильтровались (например, были слишком короткие), ставим им 0.5 (т.е. нейтрально)
users_df['positive_posts_percent'] = users_df['positive_posts_percent'].fillna(0.5)

print(users_df[['user_id', 'self_posts_num', 'positive_posts_percent']].head())

     user_id  self_posts_num  positive_posts_percent
0  230173495              11                1.000000
1   33545305              58                0.760000
2  104916144              98                0.948980
3   42020745              97                0.808081
4  641099221              91                0.862069
